# Розділ 7: Створення чат-додатків
## Швидкий старт з OpenAI API

Цей ноутбук адаптований з [репозиторію прикладів Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst), який включає ноутбуки для доступу до сервісів [Azure OpenAI](aoai-assignment.ipynb).

Python OpenAI API також працює з моделями Azure OpenAI, з деякими модифікаціями. Дізнайтеся більше про відмінності тут: [Як переключатися між кінцевими точками OpenAI і Azure OpenAI за допомогою Python](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Огляд  
"Великі мовні моделі — це функції, які відображають текст у текст. За заданим вхідним рядком тексту велика мовна модель намагається передбачити текст, який з’явиться наступним"(1). Цей блокнот "quickstart" познайомить користувачів із основними поняттями LLM, основними вимогами пакету для початку роботи з AML, м’яким вступом до проєктування підказок і кількома короткими прикладами різних варіантів використання. 


## Зміст  

[Огляд](#overview)  
[Як користуватися сервісом OpenAI](#how-to-use-openai-service)  
[1. Створення вашого сервісу OpenAI](#1.-creating-your-openai-service)  
[2. Встановлення](#2.-installation)    
[3. Облікові дані](#3.-credentials)  

[Випадки використання](#use-cases)    
[1. Підсумувати текст](#1.-summarize-text)  
[2. Класифікувати текст](#2.-classify-text)  
[3. Генерувати нові назви продуктів](#3.-generate-new-product-names)  
[4. Тонке налаштування класифікатора](#4.fine-tune-a-classifier)  

[Посилання](#references)


### Створіть свій перший запит  
Це коротке завдання надасть базове введення у подачу запитів до моделі OpenAI для простої задачі "реферування".  


**Кроки**:  
1. Встановіть бібліотеку OpenAI у вашому середовищі Python  
2. Завантажте стандартні допоміжні бібліотеки та встановіть звичні облікові дані безпеки OpenAI для створеного вами сервісу OpenAI  
3. Виберіть модель для вашого завдання  
4. Створіть простий запит для моделі  
5. Надішліть ваш запит до API моделі!  


### 1. Встановлення OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Імпортуйте допоміжні бібліотеки та створіть облікові дані


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Пошук відповідної моделі  
Моделі, такі як GPT-4o та GPT-4o mini, можуть розуміти та генерувати природну мову і доступні на платформі OpenAI з різними рівнями потужності та швидкості, підходящими для різних завдань.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. Дизайн підказок  

"Магія великих мовних моделей полягає в тому, що, тренуючись мінімізувати цю помилку прогнозування на величезних обсягах тексту, моделі в кінцевому підсумку вивчають концепції, корисні для цих прогнозів. Наприклад, вони вивчають такі поняття"(1):

* як писати слова
* як працює граматика
* як перефразовувати
* як відповідати на запитання
* як підтримувати бесіду
* як писати багатьма мовами
* як програмувати
* тощо.

#### Як керувати великою мовною моделлю  
"З усіх вхідних даних до великої мовної моделі найбільший вплив має текстова підказка(1).

Великі мовні моделі можна стимулювати до створення відповідей кількома способами:

Інструкція: скажіть моделі, що ви хочете
Завершення: спонукайте модель доповнити початок того, що ви хочете
Демонстрація: покажіть моделі, що ви хочете, одним із способів:
Кілька прикладів у підказці
Декілька сотень або тисяч прикладів у навчальному наборі для донавчання"



#### Існують три основні рекомендації щодо створення підказок:

**Показуйте і пояснюйте**. Чітко вказуйте, чого ви хочете, через інструкції, приклади або їх поєднання. Якщо ви хочете, щоб модель впорядкувала список предметів в алфавітному порядку або класифікувала абзац за настроєм, покажіть їй, що саме ви хочете.

**Надавайте якісні дані**. Якщо ви намагаєтесь створити класифікатор або попросити модель слідувати певному шаблону, переконайтеся, що є достатньо прикладів. Обов’язково перевіряйте свої приклади — модель зазвичай достатньо розумна, щоб помічати базові орфографічні помилки і давати відповідь, але вона також може припустити, що це зроблено навмисно, і це вплине на відповідь.

**Перевіряйте налаштування.** Параметри temperature і top_p контролюють, наскільки модель детермінована при генерації відповіді. Якщо ви просите відповідь, в якій є лише одна правильна відповідь, тоді бажано встановити їх на нижчі значення. Якщо ж ви шукаєте більш різноманітні відповіді, варто встановити їх на вищі значення. Найпоширеніша помилка користувачів із цими налаштуваннями — вважати, що вони керують «розумністю» чи «креативністю».


Джерело: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Надіслати!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Повторіть той самий виклик, як результати порівнюються? 


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Резюмувати текст  
#### Виклик  
Резюмуйте текст, додаючи «tl;dr:» в кінці текстового фрагмента. Зверніть увагу, як модель розуміє, як виконувати ряд завдань без додаткових інструкцій. Ви можете експериментувати з більш описовими запитами, ніж tl;dr, щоб змінити поведінку моделі та налаштувати отримуване резюме(3).  

Останні роботи продемонстрували значні покращення у багатьох завданнях NLP і бенчмарках шляхом попереднього навчання на великому корпусі текстів, а потім донавчання на конкретному завданні. Хоча, як правило, архітектура не залежить від конкретного завдання, цей метод все ж потребує донавчання на специфічних для завдання датасетах, що містять тисячі або десятки тисяч прикладів. Натомість люди зазвичай можуть виконувати нове мовне завдання лише на основі декількох прикладів або простих інструкцій — це те, з чим сучасні NLP-системи все ще значною мірою борються. Тут ми показуємо, що масштабування мовних моделей суттєво покращує універсальну, малообразну продуктивність, іноді навіть досягаючи конкурентоспроможності з попередніми передовими підходами з донавчання. 



Коротко


# Вправи для кількох випадків використання  
1. Підсумувати текст  
2. Класифікувати текст  
3. Генерувати нові назви продуктів


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Класифікація тексту  
#### Завдання  
Класифікуйте елементи за категоріями, що надаються під час інференсу. У наведеному нижче прикладі ми надаємо як категорії, так і текст для класифікації у підказці (*playground_reference). 

Запит клієнта: Привіт, нещодавно зламалась одна з клавіш на клавіатурі мого ноутбука, і мені потрібна заміна:

Класифікована категорія:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Створення нових назв продуктів
#### Виклик
Створіть назви продуктів на основі прикладів слів. Тут ми включаємо в підказку інформацію про продукт, для якого збираємося згенерувати назви. Ми також надаємо схожий приклад, щоб показати бажаний шаблон. Ми також встановили високе значення температури, щоб збільшити випадковість і отримати більш інноваційні відповіді.

Опис продукту: домашній міксер для молочних коктейлів
Початкові слова: швидкий, корисний, компактний.
Назви продуктів: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Опис продукту: пара взуття, що підходить для будь-якого розміру ноги.
Початкові слова: адаптивний, підходить, омні-фіт.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Посилання  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Портал Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Кращі практики для тонкого налаштування моделей GPT для класифікації тексту](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Для додаткової допомоги  
[Команда комерціалізації OpenAI](AzureOpenAITeam@microsoft.com) 


# Учасники
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
